# qreg_consump

> 对应代码：`EconometricsCode/code_in_notes/Chap.4/qreg_consump.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.4`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.4")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们使用 2017 年 CHFS 家庭数据，剔除收入为负以及消费或收入被截尾（censor 标记非零）的观测，然后生成消费与收入的对数（加 1 避免零值）及收入对数的平方项。

In [ ]:
%%stata
use ../datasets/chfs2017_hh.dta, clear
drop if total_income < 0
drop if max(censor_total_consump, censor_total_income)
gen log_total_consump = log(total_consump+1)
gen log_total_income = log(total_income+1)
gen log_total_income2 = log_total_income^2

我们先看线性设定：qreg 默认估计中位数回归（q=0.5），即最小化绝对偏差和，得到条件中位数函数；再用 q(0.25)、q(0.75) 分别估计 25% 与 75% 条件分位数。比较三条分位数回归线的斜率，可以判断消费-收入关系（对数意义下的弹性）在条件分布不同位置是否相同——这是分位数回归相对 OLS 的核心优势。

In [ ]:
%%stata
// 一次函数
qreg log_total_consump log_total_income
predict pred_consump_50
label variable pred_consump_50 "50%分位数"
qreg log_total_consump log_total_income, q(0.25)
predict pred_consump_25
label variable pred_consump_25 "25%分位数"
qreg log_total_consump log_total_income, q(0.75)
predict pred_consump_75
label variable pred_consump_75 "75%分位数"

按收入排序后把散点与三条分位数拟合线叠加：若三条线近似平行，说明收入对消费的影响在分布各分位点上相同（仅位置移动）；若斜率不同，则存在异质的边际倾向。

In [ ]:
%%stata
sort total_income
twoway (scatter log_total_consump log_total_income) (line pred_consump_50 log_total_income) (line pred_consump_25 log_total_income) (line pred_consump_75 log_total_income)

再加入收入对数的平方项，允许分位数函数为收入的二次函数：log_total_income* 利用 Stata 通配符同时纳入水平项与平方项。这样可以考察消费-收入关系是否非线性，以及非线性的形状是否随分位点变化。

In [ ]:
%%stata
// 二次函数
qreg log_total_consump log_total_income*
predict pred_consump2_50
label variable pred_consump_50 "50%分位数"
qreg log_total_consump log_total_income*, q(0.25)
predict pred_consump2_25
label variable pred_consump_25 "25%分位数"
qreg log_total_consump log_total_income*, q(0.75)
predict pred_consump2_75
label variable pred_consump_75 "75%分位数"

同样按收入排序后画出二次设定下的三条分位数拟合曲线，与一次设定的图对照。

In [ ]:
%%stata
sort total_income
twoway (scatter log_total_consump log_total_income) (line pred_consump2_50 log_total_income) (line pred_consump2_25 log_total_income) (line pred_consump2_75 log_total_income)